In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

drive_training_data = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/training_data"
local_training_data = "/content/xView_local/training_data"

if not os.path.exists(local_training_data):
    print("Copie du dataset vers le disque local...")
    shutil.copytree(drive_training_data, local_training_data)
    print("Terminé.")
else:
    print("Déjà présent en local.")

for split in ["train", "val"]:
    n_img = len(os.listdir(os.path.join(local_training_data, "images", split)))
    print(f"{split} : {n_img} images")

Mounted at /content/drive
Copie du dataset vers le disque local...
Terminé.
train : 10762 images
val : 1878 images


In [2]:
import requests

url = "https://raw.githubusercontent.com/DIUx-xView/data_utilities/master/xview_class_labels.txt"
response = requests.get(url)

class_names = {}
for line in response.text.strip().split('\n'):
    idx, name = line.split(':')
    class_names[int(idx)] = name.strip()

official_ids = sorted(class_names.keys())
print(f"{len(official_ids)} classes chargées")

60 classes chargées


In [3]:
import yaml

names_dict = {i: class_names[official_id] for i, official_id in enumerate(official_ids)}

data_yaml = {
    "path": local_training_data,
    "train": "images/train",
    "val": "images/val",
    "names": names_dict
}

yaml_path = os.path.join(local_training_data, "data.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False, allow_unicode=True)

print(open(yaml_path).read())

path: /content/xView_local/training_data
train: images/train
val: images/val
names:
  0: Fixed-wing Aircraft
  1: Small Aircraft
  2: Cargo Plane
  3: Helicopter
  4: Passenger Vehicle
  5: Small Car
  6: Bus
  7: Pickup Truck
  8: Utility Truck
  9: Truck
  10: Cargo Truck
  11: Truck w/Box
  12: Truck Tractor
  13: Trailer
  14: Truck w/Flatbed
  15: Truck w/Liquid
  16: Crane Truck
  17: Railway Vehicle
  18: Passenger Car
  19: Cargo Car
  20: Flat Car
  21: Tank car
  22: Locomotive
  23: Maritime Vessel
  24: Motorboat
  25: Sailboat
  26: Tugboat
  27: Barge
  28: Fishing Vessel
  29: Ferry
  30: Yacht
  31: Container Ship
  32: Oil Tanker
  33: Engineering Vehicle
  34: Tower crane
  35: Container Crane
  36: Reach Stacker
  37: Straddle Carrier
  38: Mobile Crane
  39: Dump Truck
  40: Haul Truck
  41: Scraper/Tractor
  42: Front loader/Bulldozer
  43: Excavator
  44: Cement Mixer
  45: Ground Grader
  46: Hut/Tent
  47: Shed
  48: Building
  49: Aircraft Hangar
  50: Damaged 

In [4]:
!pip install ultralytics -q

import torch
print(f"GPU disponible : {torch.cuda.is_available()}")
print(f"Nom du GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Aucun'}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 9.6 MB/s eta 0:00:00
GPU disponible : True
Nom du GPU : Tesla T4


In [5]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [6]:
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    project="/content/xView_local/runs",
    name="xview_yolov8n"
)

Ultralytics 8.4.159 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/xView_local/training_data/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=xview_yolov8n, nbs=64, nms=None,

In [7]:
drive_runs = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/runs/xview_yolov8n"

os.makedirs(os.path.dirname(drive_runs), exist_ok=True)
shutil.copytree("/content/xView_local/runs/xview_yolov8n", drive_runs)

print("Modèle et résultats sauvegardés sur Drive.")

Modèle et résultats sauvegardés sur Drive.
